### GWAS-VCF Python parser

In [15]:
import pygwasvcf
from pathlib import *
import os, sys
import pandas as pd
import numpy as np

### VCF format

https://samtools.github.io/hts-specs/VCFv4.2.pdf



### Run tabix (an example)

- change diretory <your dir>

```Bash
sudo apt install tabix

cd ../VCF/parkinson/ukb-b-16943-sibling  
ls -ls *.vcf.gz  
tabix -p vcf ukb-b-16943.vcf.gz  
tabix -p vcf ukb-b-16943_raw.vcf.gz
```

In [16]:
root0 = Path("/media/flavio/36873e3e-7941-48d7-aecb-45900ef92659/VCF")
root_data = root0 / 'parkinson/ukb-b-16943-siblings'
files = os.listdir(root_data)
files

['ukb-b-16943_raw.vcf.gz',
 'ukb-b-16943.vcf.gz',
 'ukb-b-16943_raw.vcf.gz.tbi',
 'README.md',
 'ukb-b-16943.vcf.gz.tbi']

In [18]:
fname = files[0]
print(fname)

filename = root_data / fname


with pygwasvcf.GwasVcf(filename) as fh:
    # print dictionary of GWAS metadata
    dic_meta = fh.get_metadata()

len(dic_meta)

ukb-b-16943_raw.vcf.gz


1

In [20]:
for key, dic2 in dic_meta.items():
    print(key)
    for key2, val in dic2.items():
        print(f"\t{key} -> {key2} -> {val}")


UKB-b:16943
	UKB-b:16943 -> TotalVariants -> 9851866
	UKB-b:16943 -> VariantsNotRead -> 0
	UKB-b:16943 -> HarmonisedVariants -> 9851866
	UKB-b:16943 -> VariantsNotHarmonised -> 0
	UKB-b:16943 -> SwitchedAlleles -> 9851866
	UKB-b:16943 -> TotalControls -> 359194
	UKB-b:16943 -> TotalCases -> 2005
	UKB-b:16943 -> StudyType -> CaseControl


In [50]:
def simplify(x):
    x = eval(x)
    x = list(x)

    if len(x) == 1:
        return x[0]
    return x 


with pygwasvcf.GwasVcf(filename) as fh:
    i=0
    for variant in fh.query(contig="1", start=1, stop=1_000_000):
        print(f"id: {variant.id:12}, chr: {variant.chrom:2}, pos: {variant.pos:10}, ref: {variant.ref}, alts: {simplify(variant.alts.__str__())}, contig: {variant.contig}")
        i+=1
        if i == 5: break

     

id: rs10399793  , chr: 1 , pos:      49298, ref: T, alts: C, contig: 1
id: rs2462492   , chr: 1 , pos:      54676, ref: C, alts: T, contig: 1
id: rs114608975 , chr: 1 , pos:      86028, ref: T, alts: C, contig: 1
id: rs6702460   , chr: 1 , pos:      91536, ref: G, alts: T, contig: 1
id: rs8179466   , chr: 1 , pos:     234313, ref: C, alts: T, contig: 1


In [ ]:
def simplify(x):
    x = eval(x)
    x = list(x)

    if len(x) == 1:
        return x[0]
    return x 

def dump_variant(rec):
    print("CHROM:", rec.chrom)
    print("POS:", rec.pos)
    print("ID:", rec.id)
    print("REF:", rec.ref)
    print("ALTS:", simplify(rec.alts))
    print("QUAL:", rec.qual)
    print("FILTER:", list(rec.filter.keys()))

    print("\nINFO:")
    for k, v in rec.info.items():
        print(f"  {k}: {v}")

    print("\nSAMPLES:")
    for s in rec.samples:
        print(f"  {s}:")
        for k, v in rec.samples[s].items():
            print(f"    {k}: {v}")


with pygwasvcf.GwasVcf(filename) as fh:
    i=0
    for variant in fh.query(contig="1", start=1, stop=1_000_000):
        print(f"id: {variant.id:12}, chr: {variant.chrom:2}, pos: {variant.pos:10}, ref: {variant.ref}, alts: {simplify(variant.alts.__str__())}, contig: {variant.contig}")
        i+=1
        if i == 5: break

     

In [49]:
variant.info.items()

[('AF', (0.0012110000243410468,))]

In [25]:
for key, value in variant.info.items():
    print(key, value)

AF (0.07454600185155869,)


In [12]:
with pygwasvcf.GwasVcf(filename) as fh:
    print(fh.header.samples)

[W::hts_idx_load3] The index file is older than the data file: /media/flavio/36873e3e-7941-48d7-aecb-45900ef92659/VCF/parkinson/ukb-b-16943-siblings/ukb-b-16943_raw.vcf.gz.tbi


AttributeError: 'GwasVcf' object has no attribute 'header'

In [9]:
with pygwasvcf.GwasVcf(filename) as fh:
    # query by chromosome and position interval
    for variant in fh.query(contig="1", start=1, stop=1):
        # print variant-trait P value
        print(pygwasvcf.VariantRecordGwasFuns.get_pval(variant, "trait_name"))
        # print variant-trait SE
        print(pygwasvcf.VariantRecordGwasFuns.get_se(variant, "trait_name"))
        # print variant-trait beta
        print(pygwasvcf.VariantRecordGwasFuns.get_beta(variant, "trait_name"))
        # print variant-trait allele frequency
        print(pygwasvcf.VariantRecordGwasFuns.get_af(variant, "trait_name"))
        # print variant-trait ID
        print(pygwasvcf.VariantRecordGwasFuns.get_id(variant, "trait_name"))
        # create and print ID on-the-fly if missing
        print(pygwasvcf.VariantRecordGwasFuns.get_id(variant, "trait_name", create_if_missing=True))
        # print variant-trait sample size
        print(pygwasvcf.VariantRecordGwasFuns.get_ss(variant, "trait_name"))
        # print variant-trait total sample size from header if per-variant is missing
        print(pygwasvcf.VariantRecordGwasFuns.get_ss(variant, "trait_name", g.get_metadata()))
        # print variant-trait number of cases
        print(pygwasvcf.VariantRecordGwasFuns.get_nc(variant, "trait_name"))
        # print variant-trait total cases from header if per-variant is missing
        print(pygwasvcf.VariantRecordGwasFuns.get_nc(variant, "trait_name", g.get_metadata()))

[W::hts_idx_load3] The index file is older than the data file: /media/flavio/36873e3e-7941-48d7-aecb-45900ef92659/VCF/parkinson/ukb-b-16943-siblings/ukb-b-16943_raw.vcf.gz.tbi
